# Fase 1B — Preprocesamiento para Transformers (IroSvA)

Este notebook prepara la entrada común de BETO, DistilBETO y RoBERTuito.

Los tres Transformers reciben **el mismo texto**, `MESSAGE_TRANSFORMER`, generado con el preprocesamiento oficial para redes sociales en español de Pérez et al. (2022), `pysentimiento.preprocessing.preprocess_tweet`. Así, las diferencias de rendimiento entre los tres modelos se deben al modelo y no a cómo se limpió el texto.

Se mantiene el control de calidad interno del train (duplicados exactos y conflictos de etiqueta) y el diagnóstico de tokenización. Las 9 variables lingüísticas no se calculan aquí, porque los modelos de deep learning de este trabajo solo utilizan el texto; se extraen únicamente en el preprocesamiento de ML.

El test se procesa con las mismas reglas, pero **no se utiliza para eliminar registros del train**.

La prevención de leakage entre folds se realiza posteriormente, en los notebooks de cada Transformer, mediante grupos de textos coincidentes + `StratifiedGroupKFold`.

## 1. Librerías y entorno

Importaciones, versiones de los paquetes y disponibilidad de `transformers` y `pysentimiento` (este último es obligatorio).

In [6]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import emoji
from IPython.display import display

try:
    from transformers import AutoTokenizer

    TRANSFORMERS_DISPONIBLE = True
except ImportError:
    AutoTokenizer = None
    TRANSFORMERS_DISPONIBLE = False

try:
    from pysentimiento.preprocessing import preprocess_tweet

    PYSENTIMIENTO_DISPONIBLE = True
except ImportError:
    preprocess_tweet = None
    PYSENTIMIENTO_DISPONIBLE = False

VERSIONES = {"pandas": pd.__version__, "numpy": np.__version__, "emoji": emoji.__version__}
if TRANSFORMERS_DISPONIBLE:
    import transformers

    VERSIONES["transformers"] = transformers.__version__
if PYSENTIMIENTO_DISPONIBLE:
    from importlib.metadata import PackageNotFoundError, version

    try:
        VERSIONES["pysentimiento"] = version("pysentimiento")
    except PackageNotFoundError:
        VERSIONES["pysentimiento"] = "desconocida"
print("Versiones:", " | ".join(f"{k} {v}" for k, v in VERSIONES.items()))
print("transformers disponible:", TRANSFORMERS_DISPONIBLE)
print("pysentimiento disponible:", PYSENTIMIENTO_DISPONIBLE)

Versiones: pandas 3.0.3 | numpy 1.26.4 | emoji 2.15.0 | transformers 5.17.0 | pysentimiento 0.7.3
transformers disponible: True
pysentimiento disponible: True


## 2. Configuración

Las rutas, las variantes dialectales y los checkpoints se definen una sola vez en esta celda. Los CSV se exportan en `../data/transformers/`.

`ROBERTUITO_ALTERNATIVO` (`robertuito-base-deacc`) solo se registra como referencia; en este notebook no se utiliza.

In [7]:
DATA_DIR = Path("../data")
OUT_DIR = DATA_DIR / "transformers"
OUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANTES = ["mx", "es", "cu"]

CHECKPOINTS = {
    "BETO": "dccuchile/bert-base-spanish-wwm-cased",
    "DistilBETO": "dccuchile/distilbert-base-spanish-uncased",
    "RoBERTuito": "pysentimiento/robertuito-base-uncased",
}
ROBERTUITO_ALTERNATIVO = "pysentimiento/robertuito-base-deacc"

print(f"[CONFIG][Preprocesamiento DL] variantes={VARIANTES}")
print(f"[CONFIG][Preprocesamiento DL] entrada={DATA_DIR.resolve()}")
print(f"[CONFIG][Preprocesamiento DL] salida={OUT_DIR.resolve()}")
for modelo, checkpoint in CHECKPOINTS.items():
    print(f"[CONFIG][Preprocesamiento DL] {modelo:12} -> {checkpoint}")
print(f"[CONFIG][Preprocesamiento DL] {'RoBERTuito alt.':12} -> {ROBERTUITO_ALTERNATIVO}")

[CONFIG][Preprocesamiento DL] variantes=['mx', 'es', 'cu']
[CONFIG][Preprocesamiento DL] entrada=C:\Users\I21308\Documents\DeteccionDeSarcasmo\data
[CONFIG][Preprocesamiento DL] salida=C:\Users\I21308\Documents\DeteccionDeSarcasmo\data\transformers
[CONFIG][Preprocesamiento DL] BETO         -> dccuchile/bert-base-spanish-wwm-cased
[CONFIG][Preprocesamiento DL] DistilBETO   -> dccuchile/distilbert-base-spanish-uncased
[CONFIG][Preprocesamiento DL] RoBERTuito   -> pysentimiento/robertuito-base-uncased
[CONFIG][Preprocesamiento DL] RoBERTuito alt. -> pysentimiento/robertuito-base-deacc


## 3. Carga y auditoría del train oficial

Se cargan las tres variantes dialectales por separado.

Antes de eliminar registros se verifica, comparando el `MESSAGE` original de forma exacta:

- cuántos textos `MESSAGE` están repetidos;
- si un mismo texto repetido tiene **la misma etiqueta**;
- si existe un **conflicto de anotación** (mismo `MESSAGE` con etiquetas distintas).

Después de la auditoría:

1. los textos con etiquetas contradictorias se retiran **completamente** del train y se guardan en `conflictos_etiqueta_train.csv`;
2. de los duplicados restantes (misma etiqueta) se conserva una sola copia (la primera).

El test no interviene en este control. El mismo criterio se aplica en los preprocesamientos de ML y de Transformers.

In [8]:
def auditar_duplicados_train(df, variante):
    """Cuenta textos repetidos y conflictos de etiqueta dentro del train de una variante."""
    grupos = df.groupby("MESSAGE", dropna=False)["IS_IRONIC"].agg(["size", "nunique"]).reset_index()
    repetidos = grupos[grupos["size"] > 1]
    conflictos = repetidos[repetidos["nunique"] > 1]
    mismas_etiquetas = repetidos[repetidos["nunique"] == 1]

    return {
        "variante": variante,
        "n_original": len(df),
        "n_textos_repetidos": len(repetidos),
        "n_duplicados_misma_etiqueta": len(mismas_etiquetas),
        "n_conflictos_etiqueta": len(conflictos),
        "mensajes_conflicto": set(conflictos["MESSAGE"].tolist()),
    }


def cargar_train():
    """Carga el train de cada variante, retira conflictos y deduplica textos exactos."""
    frames = []
    auditoria = []
    conflictos_detalle = []

    for variante in VARIANTES:
        df = pd.read_csv(DATA_DIR / f"irosva.{variante}.training.csv")
        df["VARIANTE"] = variante

        info = auditar_duplicados_train(df, variante)
        mensajes_conflicto = info["mensajes_conflicto"]

        # Guardar TODAS las filas implicadas en conflictos.
        df_conflictos = df[df["MESSAGE"].isin(mensajes_conflicto)].copy()
        if not df_conflictos.empty:
            df_conflictos["MOTIVO_EXCLUSION"] = "mismo_texto_etiquetas_contradictorias"
            conflictos_detalle.append(df_conflictos)
            print(f"\n{variante.upper()} — conflictos de etiqueta:")
            display(
                df_conflictos[["ID", "MESSAGE", "IS_IRONIC", "VARIANTE"]].sort_values("MESSAGE")
            )

        # 1. Retirar completamente los textos con etiquetas contradictorias.
        df_sin_conflictos = df[~df["MESSAGE"].isin(mensajes_conflicto)].copy()

        # 2. En los duplicados restantes la etiqueta coincide: conservar una copia.
        df_limpio = df_sin_conflictos.drop_duplicates(subset="MESSAGE", keep="first").reset_index(
            drop=True
        )
        n_duplicados_retirados = len(df_sin_conflictos) - len(df_limpio)

        auditoria.append(
            {
                "variante": variante,
                "n_original": info["n_original"],
                "textos_repetidos": info["n_textos_repetidos"],
                "conflictos_texto": info["n_conflictos_etiqueta"],
                "filas_retiradas_conflicto": len(df_conflictos),
                "filas_duplicadas_misma_etiqueta_retiradas": n_duplicados_retirados,
                "n_tras_control_duplicados": len(df_limpio),
            }
        )

        print(
            f"[TRAIN][{variante.upper()}] {info['n_original']} originales | "
            f"{len(df_conflictos)} retirados por conflicto | "
            f"{n_duplicados_retirados} duplicados de misma etiqueta retirados | "
            f"{len(df_limpio)} restantes"
        )
        frames.append(df_limpio)

    df_train = pd.concat(frames, ignore_index=True)
    df_auditoria = pd.DataFrame(auditoria)
    df_conflictos_train = (
        pd.concat(conflictos_detalle, ignore_index=True) if conflictos_detalle else pd.DataFrame()
    )
    return df_train, df_auditoria, df_conflictos_train


df_train, df_auditoria_train, df_conflictos_train = cargar_train()

print("\nAUDITORÍA TRAIN — DUPLICADOS Y CONFLICTOS")
display(df_auditoria_train)

print(f"\n[TRAIN] Total tras control de duplicados/conflictos: {len(df_train)} registros")
print(df_train["VARIANTE"].value_counts().sort_index())

print("\nDistribución de clases:")
print(df_train.groupby("VARIANTE")["IS_IRONIC"].value_counts())

[TRAIN][MX] 2400 originales | 0 retirados por conflicto | 1 duplicados de misma etiqueta retirados | 2399 restantes

ES — conflictos de etiqueta:


,ID,MESSAGE,IS_IRONIC,VARIANTE
762,3bb2e28f5c0f1a0dd7fbcbed779ff957,Están pasando mas cosas ?\r\n\r\nUn hijo de ju...,1,es
916,5353523b4ac97aff03c46cb3d11d4d6a,Están pasando mas cosas ?\r\n\r\nUn hijo de ju...,0,es


[TRAIN][ES] 2400 originales | 2 retirados por conflicto | 1 duplicados de misma etiqueta retirados | 2397 restantes
[TRAIN][CU] 2400 originales | 0 retirados por conflicto | 0 duplicados de misma etiqueta retirados | 2400 restantes

AUDITORÍA TRAIN — DUPLICADOS Y CONFLICTOS


,variante,n_original,textos_repetidos,conflictos_texto,filas_retiradas_conflicto,filas_duplicadas_misma_etiqueta_retiradas,n_tras_control_duplicados
0,mx,2400,1,0,0,1,2399
1,es,2400,2,1,2,1,2397
2,cu,2400,0,0,0,0,2400



[TRAIN] Total tras control de duplicados/conflictos: 7196 registros
VARIANTE
cu    2400
es    2397
mx    2399
Name: count, dtype: int64

Distribución de clases:
VARIANTE  IS_IRONIC
cu        0            1600
          1             800
es        0            1598
          1             799
mx        0            1599
          1             800
Name: count, dtype: int64


## 4. Carga del test oficial

El test se carga para generar las mismas representaciones que el train (`MESSAGE_TRANSFORMER`).

No se utiliza para modificar la composición del train.

In [9]:
def cargar_test(variante):
    """Une los textos del test con sus etiquetas oficiales."""
    df_texto = pd.read_csv(DATA_DIR / f"irosva.{variante} - irosva.{variante}.test.csv")
    df_truth = pd.read_csv(DATA_DIR / f"irosva.{variante}.test.truth.csv")

    if "IS_IRONIC" in df_texto.columns:
        df_texto = df_texto.drop(columns=["IS_IRONIC"])

    df = df_texto.merge(df_truth[["ID", "IS_IRONIC"]], on="ID", how="inner")
    assert len(df) == len(df_texto), f"{variante}: hay textos del test sin etiqueta oficial"

    df["VARIANTE"] = variante
    return df


df_test = pd.concat([cargar_test(v) for v in VARIANTES], ignore_index=True)

print(f"[TEST] Total: {len(df_test)} registros")
print(df_test["VARIANTE"].value_counts().sort_index())

[TEST] Total: 1800 registros
VARIANTE
cu    600
es    600
mx    600
Name: count, dtype: int64


## 5. Preprocesamiento común para los Transformers

Se aplica directamente el preprocesamiento oficial de Pérez et al. (2022), en lugar de replicar su lógica a mano:

```python
pysentimiento.preprocessing.preprocess_tweet(texto, lang="es", shorten=3)
```

Con sus opciones por defecto para español:

- mención → `@usuario`; URL → `url`;
- hashtag → `hashtag` + su contenido separado en palabras y en minúsculas (`#NoMasCorrupcion` → `hashtag no mas corrupcion`);
- emoji → su descripción en español, delimitada como `emoji … emoji`;
- cualquier carácter repetido 3 o más veces → 3;
- risas en minúsculas normalizadas (`jajajaja` → `jaja`);
- caracteres especiales reemplazados por equivalentes (por ejemplo, comillas tipográficas → comillas rectas; el carácter `…` se elimina, mientras que `...` se conserva).

Se conservan las mayúsculas (salvo dentro de los hashtags), tildes, puntuación, `¿?` / `¡!` y stopwords. No se aplica stemming ni lematización: los Transformers tokenizan en subpalabras y usan la secuencia completa como contexto.

**Por qué un solo preprocesamiento para los tres.** Es el que Pérez et al. (2022) usan para RoBERTuito y para evaluar BETO en tuits, y la normalización agresiva no mejoró los resultados en trabajos previos citados por ellos (BERTweet). Al dar a los tres modelos exactamente el mismo texto, la comparación aísla el efecto del modelo (y de su preentrenamiento). La exploración de representaciones alternativas se mantiene en ML (`normal` / `stem` / `lemma`), donde la bolsa de palabras sí depende de cómo se normaliza el vocabulario.

**Variante cubana.** En IroSvA, MX y ES son tuits y CU son comentarios de noticias. Se aplica el mismo preprocesamiento a las tres variantes para no introducir decisiones específicas por variante: las reglas propias de Twitter (menciones, hashtags) casi no se activan en los comentarios, mientras que las elongaciones, risas y emojis sí aparecen en ambos tipos de texto.

In [10]:
def preprocesar_transformer(texto):
    """Preprocesamiento oficial de Pérez et al. (2022): común a BETO, DistilBETO y RoBERTuito."""
    return preprocess_tweet(str(texto), lang="es", shorten=3)


if not PYSENTIMIENTO_DISPONIBLE:
    raise ImportError(
        "Falta pysentimiento, necesario para los tres Transformers. "
        "Instala con `pip install pysentimiento` o `uv add pysentimiento`."
    )

## 6. Generación del texto para Transformers

Se aplica el mismo preprocesamiento al train y al test. El notebook se detiene si algún texto queda vacío.

In [11]:
for nombre, df in [("Train", df_train), ("Test", df_test)]:
    print(f"Procesando {nombre}...")
    df["MESSAGE_TRANSFORMER"] = df["MESSAGE"].fillna("").astype(str).apply(preprocesar_transformer)
    vacios = int(df["MESSAGE_TRANSFORMER"].str.strip().eq("").sum())
    if vacios:
        raise ValueError(f"{nombre}: {vacios} textos quedaron vacíos tras el preprocesamiento.")

print("\nEjemplos por variante:")
for variante in VARIANTES:
    row = df_train[df_train["VARIANTE"] == variante].iloc[0]
    print(f"\n--- {variante.upper()} ---")
    print("ORIGINAL:   ", row["MESSAGE"])
    print("TRANSFORMER:", row["MESSAGE_TRANSFORMER"])

Procesando Train...
Procesando Test...

Ejemplos por variante:

--- MX ---
ORIGINAL:    Rica económicamente, pero muy pobre en objetividad.
TRANSFORMER: Rica económicamente, pero muy pobre en objetividad.

--- ES ---
ORIGINAL:    @ArmandoRuido007 @ANTI_MERMA50 @JoanTarda En vez de Joan Tarda van a llamarle “No han tarda” en callarle la boca 🤣🤣
TRANSFORMER: @usuario @usuario @usuario En vez de Joan Tarda van a llamarle "No han tarda" en callarle la boca  emoji cara revolviéndose de la risa emoji  emoji cara revolviéndose de la risa emoji

--- CU ---
ORIGINAL:    magnifico
TRANSFORMER: magnifico


## 7. Diagnóstico de tokenización — solo train

Para cada tokenizer (BETO, DistilBETO y RoBERTuito) y variante dialectal, sobre `MESSAGE_TRANSFORMER`, se calculan estadísticas descriptivas de longitud: percentiles 50/95/99, máximo, % de textos con más de 64 y 128 tokens, tokens por palabra y `[UNK]` por cada 1000 tokens.

Este diagnóstico utiliza únicamente el train y no modifica registros.

In [12]:
def cargar_tokenizers():
    """Carga los tokenizers oficiales de cada checkpoint."""
    if not TRANSFORMERS_DISPONIBLE:
        raise ImportError(
            "Falta transformers. Instala con `pip install transformers` o `uv add transformers`."
        )
    return {
        modelo: AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
        for modelo, checkpoint in CHECKPOINTS.items()
    }


def calcular_stats_tokenizacion(tokenizer, textos):
    """Estadísticas de longitud en tokens para una lista de textos."""
    longitudes = []
    unk = 0
    total_tokens = 0
    total_palabras = 0
    unk_id = tokenizer.unk_token_id

    for texto in textos:
        texto = str(texto)
        ids = tokenizer(texto, add_special_tokens=True, truncation=False, padding=False)[
            "input_ids"
        ]

        longitudes.append(len(ids))
        total_tokens += len(ids)
        total_palabras += max(len(texto.split()), 1)
        if unk_id is not None:
            unk += sum(token == unk_id for token in ids)

    arr = np.asarray(longitudes, dtype=np.int32)
    return {
        "n": len(arr),
        "p50": np.percentile(arr, 50),
        "p95": np.percentile(arr, 95),
        "p99": np.percentile(arr, 99),
        "max": int(arr.max()),
        "pct_gt_64": 100 * (arr > 64).mean(),
        "pct_gt_128": 100 * (arr > 128).mean(),
        "tokens_por_palabra": total_tokens / max(total_palabras, 1),
        "unk_por_1000_tokens": 1000 * unk / max(total_tokens, 1),
    }


def diagnosticar_train():
    """Aplica el diagnóstico a cada tokenizer × variante sobre MESSAGE_TRANSFORMER."""
    tokenizers = cargar_tokenizers()
    filas = []
    for tokenizer_name, tok in tokenizers.items():
        cfg, columna = tokenizer_name.upper(), "MESSAGE_TRANSFORMER"
        for variante in VARIANTES:
            textos = (
                df_train.loc[df_train["VARIANTE"] == variante, columna]
                .fillna("")
                .astype(str)
                .tolist()
            )
            filas.append(
                {
                    "configuracion": cfg,
                    "tokenizer": tokenizer_name,
                    "variante": variante,
                    **calcular_stats_tokenizacion(tok, textos),
                }
            )
    return pd.DataFrame(filas)


if TRANSFORMERS_DISPONIBLE:
    df_tokenizacion = diagnosticar_train()
    display(
        df_tokenizacion.style.format(
            {
                "p50": "{:.1f}",
                "p95": "{:.1f}",
                "p99": "{:.1f}",
                "pct_gt_64": "{:.2f}%",
                "pct_gt_128": "{:.2f}%",
                "tokens_por_palabra": "{:.3f}",
                "unk_por_1000_tokens": "{:.3f}",
            }
        )
    )
else:
    df_tokenizacion = None
    print("Instala transformers para ejecutar el diagnóstico.")

,configuracion,tokenizer,variante,n,p50,p95,p99,max,pct_gt_64,pct_gt_128,tokens_por_palabra,unk_por_1000_tokens
0,BETO,BETO,mx,2399,27.0,68.0,89.0,193,6.67%,0.17%,1.476,0.943
1,BETO,BETO,es,2397,37.0,76.0,100.1,188,13.18%,0.25%,1.559,3.642
2,BETO,BETO,cu,2400,38.0,70.0,88.0,124,10.04%,0.00%,1.413,0.463
3,DISTILBETO,DistilBETO,mx,2399,26.0,64.0,78.0,180,4.71%,0.08%,1.410,0.986
4,DISTILBETO,DistilBETO,es,2397,35.0,73.0,94.0,156,11.39%,0.21%,1.493,3.803
5,DISTILBETO,DistilBETO,cu,2400,36.0,66.0,74.0,84,6.04%,0.00%,1.331,0.491
6,ROBERTUITO,RoBERTuito,mx,2399,24.0,60.0,70.0,148,2.79%,0.04%,1.305,0.000
7,ROBERTUITO,RoBERTuito,es,2397,33.0,70.0,88.0,150,8.72%,0.13%,1.405,0.000
8,ROBERTUITO,RoBERTuito,cu,2400,35.0,64.0,69.0,77,4.25%,0.00%,1.276,0.000


## 8. Exportación final por variante

Se exportan 6 archivos:

- 3 variantes dialectales (`mx`, `es`, `cu`);
- 2 splits (`train`, `test`).

Cada archivo contiene los identificadores, la etiqueta, el texto original y `MESSAGE_TRANSFORMER`; es la entrada común de BETO, DistilBETO y RoBERTuito. También se guardan el diagnóstico de tokenización, la auditoría del train, las filas retiradas por conflicto y un manifest con el número de filas por variante.

El preprocesamiento no elimina registros del train en función del contenido del test.

**Versión del pipeline:** `transformers_comun_v2` (un solo texto común para los tres Transformers; la versión anterior, `v1`, generaba las columnas BASE, SOCIAL y ROBERTUITO). No se retiran del train los textos que coinciden con el test; esas coincidencias se reportan después, en la evaluación final, como análisis de sensibilidad.

In [13]:
COLUMNAS_BASE = ["ID", "TOPIC", "IS_IRONIC", "MESSAGE", "VARIANTE"]

COLUMNAS_TEXTO = ["MESSAGE_TRANSFORMER"]


def exportar_por_variante(df, split):
    """Exporta un CSV por variante con las columnas base y el texto común de los Transformers."""
    for variante in VARIANTES:
        salida = df.loc[df["VARIANTE"] == variante, COLUMNAS_BASE + COLUMNAS_TEXTO].reset_index(
            drop=True
        )
        ruta = OUT_DIR / f"{split}_transformers_{variante}.csv"
        salida.to_csv(ruta, index=False)
        print(f"[EXPORT][{split.upper()}][{variante.upper()}] {len(salida)} registros -> {ruta}")


print("=" * 78)
print("EXPORTACIÓN FINAL — TRANSFORMERS")
print("=" * 78)

for split, df in [("train", df_train), ("test", df_test)]:
    exportar_por_variante(df, split)

if df_tokenizacion is not None:
    ruta_diag = OUT_DIR / "diagnostico_tokenizacion_train.csv"
    df_tokenizacion.to_csv(ruta_diag, index=False)
    print(f"\n[EXPORT][AUDIT] Diagnóstico de tokenización -> {ruta_diag}")


# Auditoría interna del train.
df_auditoria_train.to_csv(OUT_DIR / "auditoria_duplicados_train.csv", index=False)

if len(df_conflictos_train) > 0:
    df_conflictos_train.to_csv(OUT_DIR / "conflictos_etiqueta_train.csv", index=False)


# Manifest mínimo: permite comprobar versión y cantidad de filas.
manifest = pd.DataFrame(
    [
        {
            "pipeline_version": "transformers_comun_v2",
            "variante": variante,
            "train_final": int((df_train["VARIANTE"] == variante).sum()),
            "test_n": int((df_test["VARIANTE"] == variante).sum()),
        }
        for variante in VARIANTES
    ]
)
manifest.to_csv(OUT_DIR / "manifest_preprocesamiento_transformers.csv", index=False)

print("\nMANIFEST DEL PREPROCESAMIENTO")
display(manifest)

print("\n[PREPROCESS][Transformers] EXPORTACIÓN: OK")

EXPORTACIÓN FINAL — TRANSFORMERS
[EXPORT][TRAIN][MX] 2399 registros -> ..\data\transformers\train_transformers_mx.csv
[EXPORT][TRAIN][ES] 2397 registros -> ..\data\transformers\train_transformers_es.csv
[EXPORT][TRAIN][CU] 2400 registros -> ..\data\transformers\train_transformers_cu.csv
[EXPORT][TEST][MX] 600 registros -> ..\data\transformers\test_transformers_mx.csv
[EXPORT][TEST][ES] 600 registros -> ..\data\transformers\test_transformers_es.csv
[EXPORT][TEST][CU] 600 registros -> ..\data\transformers\test_transformers_cu.csv

[EXPORT][AUDIT] Diagnóstico de tokenización -> ..\data\transformers\diagnostico_tokenizacion_train.csv

MANIFEST DEL PREPROCESAMIENTO


,pipeline_version,variante,train_final,test_n
0,transformers_comun_v2,mx,2399,600
1,transformers_comun_v2,es,2397,600
2,transformers_comun_v2,cu,2400,600



[PREPROCESS][Transformers] EXPORTACIÓN: OK


## 9. Uso posterior

BETO, DistilBETO y RoBERTuito cargan directamente:

```text
data/transformers/train_transformers_{mx,es,cu}.csv
data/transformers/test_transformers_{mx,es,cu}.csv
```

Según el diseño previsto, cada modelo construye después sus propios grupos de textos coincidentes a partir de sus representaciones.

El preprocesamiento no decide los folds ni elimina registros en función del test.

## 10. Resumen metodológico

### Train

- control interno de duplicados exactos y conflictos de etiqueta;
- generación de `MESSAGE_TRANSFORMER` con `preprocess_tweet` (Pérez et al., 2022), común a los tres Transformers;
- diagnóstico de tokenización;
- exportación.

### Modelos

Según el diseño previsto, BETO, DistilBETO y RoBERTuito:

1. cargan el train preprocesado;
2. construyen sus propios grupos de textos coincidentes;
3. ejecutan `StratifiedGroupKFold` 5×3;
4. seleccionan hiperparámetros con F1-Macro.

### Test

Se mantiene como conjunto de evaluación final y no determina qué registros permanecen en train.